In [ ]:
#imports
import numpy as np
import pandas as pd
import geopandas as gpd
import holidays
from pygam import LinearGAM, s, f
from datetime import timedelta
from pathlib import Path
from tqdm import tqdm  
from rapidfuzz import process, fuzz
import os 
from quantile_forest import RandomForestQuantileRegressor
import pyarrow.parquet as pq
import pyarrow.fs as pafs

In [ ]:
#formatting
temp_wide = pd.read_parquet("/home/s13q981/HTL/ba_hourly_temp.parquet")
df_temp = (
    temp_wide.reset_index()
    .drop(columns="XTIME")
    .melt(id_vars="Time", var_name="balancing_authority_code_eia", value_name="avg_temp_K")
    .rename(columns={"Time": "datetime_utc"})
    .sort_values(["balancing_authority_code_eia", "datetime_utc"])
    .reset_index(drop=True)
)

In [ ]:
#smoothed temperature variables
alpha_95, alpha_99 = 0.05, 0.01
df_temp["temp_s95"] = df_temp.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ewm(alpha=alpha_95, adjust=False).mean()
)
df_temp["temp_s99"] = df_temp.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ewm(alpha=alpha_99, adjust=False).mean()
)

In [ ]:
# continuity check: confirm this is the FULL 2016-Sep2022 span per BA, not a partial checkpoint
print(df_temp.groupby("balancing_authority_code_eia")["datetime_utc"].agg(["min", "max", "count"]))
gaps = df_temp.groupby("balancing_authority_code_eia")["datetime_utc"].diff().value_counts()
print(gaps)

In [ ]:
#load EIA 930 via PUDL
s3 = pafs.S3FileSystem(anonymous=True, region="us-west-2")
df_y = pq.read_table(
    "pudl.catalyst.coop/nightly/core_eia930__hourly_operations.parquet",
    filesystem=s3
).to_pandas(types_mapper=pd.ArrowDtype)

df_y["datetime_utc"] = df_y["datetime_utc"].astype("datetime64[ns]")  # match df_temp's dtype for the merge key

print(df_y.shape)
print(df_y.columns.tolist())

In [ ]:
#merge with EIA
df_merged = df_y.merge(
    df_temp,   # <-- df_temp, not temp_long since this carries temp_s95/temp_s99
    on=["balancing_authority_code_eia", "datetime_utc"],
    how="inner",
)

df_merged = df_merged.sort_values(["balancing_authority_code_eia", "datetime_utc"])
df_merged["load_24"]  = df_merged.groupby("balancing_authority_code_eia")["demand_adjusted_mwh"].shift(24)
df_merged["load_168"] = df_merged.groupby("balancing_authority_code_eia")["demand_adjusted_mwh"].shift(168)

print(df_merged.shape)
print(df_merged["balancing_authority_code_eia"].nunique(), "BAs after merge")
print(df_merged[["datetime_utc","balancing_authority_code_eia","demand_adjusted_mwh","avg_temp_K","load_24","load_168"]].head())

In [ ]:
#calender variables
df_merged["hour"]      = df_merged["datetime_utc"].dt.hour
df_merged["dayofweek"] = df_merged["datetime_utc"].dt.dayofweek
df_merged["dayofyear"] = df_merged["datetime_utc"].dt.dayofyear
us_holidays = holidays.US()
df_merged["is_holiday"] = df_merged["datetime_utc"].dt.date.apply(lambda x: x in us_holidays).astype(int)

print(df_merged.shape)
print(df_merged["balancing_authority_code_eia"].nunique(), "BAs after merge")
print(df_merged[["datetime_utc","balancing_authority_code_eia","demand_adjusted_mwh",
                  "avg_temp_K","temp_s95","temp_s99","load_24","load_168","hour","is_holiday"]].head())


In [ ]:
#interconnect / national aggregation
# Builds EASTERN / WESTERN / ERCOT / NATIONAL as four extra pseudo-BA rows and appends
# them to df_merged, so the existing GAM (below) and XGB stacking cells automatically
# fit/forecast them too -- these serve as the independent top-level base forecasts
# MinT reconciliation needs (without them, MinT collapses to simple bottom-up summing).
# BA -> interconnect mapping derived from EIA930_Reference_Tables-2.xlsx "BAs" sheet
# (Region/Country Code column), cross-checked against known NERC interconnect membership.
BA_TO_INTERCONNECT = {
    # Western (28)
    "BANC":"WESTERN","CISO":"WESTERN","IID":"WESTERN","LDWP":"WESTERN","TIDC":"WESTERN",
    "AVA":"WESTERN","BPAT":"WESTERN","CHPD":"WESTERN","DOPD":"WESTERN","GCPD":"WESTERN",
    "IPCO":"WESTERN","NEVP":"WESTERN","NWMT":"WESTERN","PACE":"WESTERN","PACW":"WESTERN",
    "PGE":"WESTERN","PSCO":"WESTERN","PSEI":"WESTERN","SCL":"WESTERN","TPWR":"WESTERN",
    "WACM":"WESTERN","WAUW":"WESTERN","AZPS":"WESTERN","EPE":"WESTERN","PNM":"WESTERN",
    "SRP":"WESTERN","TEPC":"WESTERN","WALC":"WESTERN",
    # Eastern (26)
    "AEC":"EASTERN","SOCO":"EASTERN","AECI":"EASTERN","LGEE":"EASTERN","MISO":"EASTERN",
    "CPLE":"EASTERN","CPLW":"EASTERN","DUK":"EASTERN","SC":"EASTERN","SCEG":"EASTERN",
    "FMPP":"EASTERN","FPC":"EASTERN","FPL":"EASTERN","GVL":"EASTERN","HST":"EASTERN",
    "JEA":"EASTERN","NSB":"EASTERN","SEC":"EASTERN","TAL":"EASTERN","TEC":"EASTERN",
    "ISNE":"EASTERN","NYIS":"EASTERN","PJM":"EASTERN","TVA":"EASTERN","SPA":"EASTERN","SWPP":"EASTERN",
    # ERCOT (1)
    "ERCO":"ERCOT",
}

present_bas = set(df_merged["balancing_authority_code_eia"].unique())
unmapped = present_bas - set(BA_TO_INTERCONNECT)
if unmapped:
    print(f"WARNING: {len(unmapped)} BAs in df_merged have no interconnect mapping and will be excluded from aggregation: {sorted(unmapped)}")

# static (non-time-varying) load weights, so the temperature feature never depends on
# same-hour demand -- avoids leaking the target into an aggregate-level predictor
ba_weights = df_merged.groupby("balancing_authority_code_eia")["demand_adjusted_mwh"].mean()

def build_aggregate_group(df_source, group_value_map, weights):
    frames = []
    for group_name in sorted(set(group_value_map.values())):
        bas_in_group = [ba for ba, g in group_value_map.items() if g == group_name and ba in present_bas]
        sub = df_source[df_source["balancing_authority_code_eia"].isin(bas_in_group)].copy()
        w = weights.reindex(bas_in_group).fillna(0)
        w = w / w.sum()
        sub["_w"] = sub["balancing_authority_code_eia"].map(w)
        sub["_wtemp"] = sub["avg_temp_K"] * sub["_w"]
        agg = sub.groupby("datetime_utc").agg(
            demand_adjusted_mwh=("demand_adjusted_mwh", "sum"),
            avg_temp_K=("_wtemp", "sum"),
        ).reset_index()
        agg["balancing_authority_code_eia"] = group_name
        frames.append(agg)
    return pd.concat(frames, ignore_index=True)

df_interconnect = build_aggregate_group(df_merged, BA_TO_INTERCONNECT, ba_weights)
national_map = {ba: "NATIONAL" for ba in BA_TO_INTERCONNECT}
df_national = build_aggregate_group(df_merged, national_map, ba_weights)

df_agg = pd.concat([df_interconnect, df_national], ignore_index=True)
df_agg = df_agg.sort_values(["balancing_authority_code_eia", "datetime_utc"]).reset_index(drop=True)

# re-smooth temperature at the aggregate level (recomputed on the aggregated series
# itself, not averaged from already-smoothed BA-level temp_s95/temp_s99)
df_agg["temp_s95"] = df_agg.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ewm(alpha=alpha_95, adjust=False).mean()
)
df_agg["temp_s99"] = df_agg.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ewm(alpha=alpha_99, adjust=False).mean()
)

# calendar variables, same logic as the BA-level cell above
df_agg["hour"]      = df_agg["datetime_utc"].dt.hour
df_agg["dayofweek"] = df_agg["datetime_utc"].dt.dayofweek
df_agg["dayofyear"] = df_agg["datetime_utc"].dt.dayofyear
df_agg["is_holiday"] = df_agg["datetime_utc"].dt.date.apply(lambda x: x in us_holidays).astype(int)

# lag features, computed within each aggregate group
df_agg = df_agg.sort_values(["balancing_authority_code_eia", "datetime_utc"])
df_agg["load_24"]  = df_agg.groupby("balancing_authority_code_eia")["demand_adjusted_mwh"].shift(24)
df_agg["load_168"] = df_agg.groupby("balancing_authority_code_eia")["demand_adjusted_mwh"].shift(168)

agg_cols = ["datetime_utc","balancing_authority_code_eia","demand_adjusted_mwh",
            "avg_temp_K","temp_s95","temp_s99","load_24","load_168","hour","dayofweek","dayofyear","is_holiday"]
df_merged["balancing_authority_code_eia"] = df_merged["balancing_authority_code_eia"].astype(str)
df_agg["balancing_authority_code_eia"] = df_agg["balancing_authority_code_eia"].astype(str)
df_merged = pd.concat([df_merged[agg_cols], df_agg[agg_cols]], ignore_index=True)
df_merged = df_merged.sort_values(["balancing_authority_code_eia", "datetime_utc"]).reset_index(drop=True)

print(df_merged["balancing_authority_code_eia"].nunique(), "groups after adding interconnect/national aggregates")
print(df_merged[df_merged["balancing_authority_code_eia"].isin(["EASTERN","WESTERN","ERCOT","NATIONAL"])]
      .groupby("balancing_authority_code_eia")["demand_adjusted_mwh"].agg(["mean","min","max","count"]))

In [ ]:
#train test split

TRAIN_START = "2016-01-01"
TRAIN_END   = "2019-12-31 23:00:00"
TEST_START  = "2020-01-01"
TEST_END    = "2022-09-30 23:00:00"

df_merged = df_merged.sort_values(["balancing_authority_code_eia", "datetime_utc"])

train_mask = (df_merged["datetime_utc"] >= TRAIN_START) & (df_merged["datetime_utc"] <= TRAIN_END)
test_mask  = (df_merged["datetime_utc"] >= TEST_START) & (df_merged["datetime_utc"] <= TEST_END)

df_train = df_merged.loc[train_mask].copy()
df_test  = df_merged.loc[test_mask].copy()

# sub-period labels within test, for reporting no separate splits, just labels
# on the one continuous test set, so COVID-period performance can be isolated later
bounds = [
    df_test["datetime_utc"] < pd.Timestamp("2020-03-01"),
    df_test["datetime_utc"] < pd.Timestamp("2020-05-01"),
    df_test["datetime_utc"] < pd.Timestamp("2021-01-01"),
]
labels = ["pre_shock", "acute_covid", "recovery"]
df_test["subperiod"] = np.select(bounds, labels, default="stable")

print("Train:", df_train["datetime_utc"].min(), "to", df_train["datetime_utc"].max(), " rows:", len(df_train))
print("Test :", df_test["datetime_utc"].min(), "to", df_test["datetime_utc"].max(), " rows:", len(df_test))
print(df_test["subperiod"].value_counts())

In [ ]:
#GAM
from pygam import LinearGAM, s, f, te

FEATURES = ["hour", "dayofweek", "dayofyear", "is_holiday",
            "load_24", "load_168", "avg_temp_K", "temp_s95", "temp_s99"]
TARGET = "demand_adjusted_mwh"

ba_data = {}  # ba_code: train/test data + fitted GAM + GAM predictions

for ba in tqdm(df_train["balancing_authority_code_eia"].unique(), desc="Fitting GAMs"):
    df_train_ba = df_train[df_train["balancing_authority_code_eia"] == ba].dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
    df_test_ba  = df_test[df_test["balancing_authority_code_eia"] == ba].dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)

    if len(df_train_ba) < 200 or len(df_test_ba) == 0:
        print(f"Skipping {ba}: insufficient data (train={len(df_train_ba)}, test={len(df_test_ba)})")
        continue

    X_train = df_train_ba[FEATURES].to_numpy(dtype=float)
    y_train = df_train_ba[TARGET].to_numpy(dtype=float)
    X_test  = df_test_ba[FEATURES].to_numpy(dtype=float)
    y_test  = df_test_ba[TARGET].to_numpy(dtype=float)

    gam = LinearGAM(
        s(0, n_splines=24, basis="cp") +   # hour, cyclic
        s(1, n_splines=7) +                 # dayofweek
        s(2, n_splines=10, basis="cp") +    # dayofyear, cyclic
        f(3) +                               # is_holiday
        s(4, n_splines=10) +                # load_24
        s(5, n_splines=10) +                # load_168
        s(6, n_splines=10) +                # avg_temp_K
        s(7, n_splines=10) +                # temp_s95
        s(8, n_splines=10) +                # temp_s99
        te(0, 6, n_splines=[8, 8])           # hour x avg_temp_K
    )
    gam.gridsearch(X_train, y_train, lam=np.logspace(-3, 3, 11), progress=False)

    ba_data[ba] = {
        "df_train": df_train_ba, "df_test": df_test_ba,
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "gam": gam,
        "gam_pred_train": gam.predict(X_train),
        "gam_pred_test": gam.predict(X_test),
    }
    print(f"{ba}: AIC={gam.statistics_['AIC']:.1f}  train_n={len(df_train_ba)}  test_n={len(df_test_ba)}")

print(f"\nFitted GAMs for {len(ba_data)} / {df_train['balancing_authority_code_eia'].nunique()} BAs")

In [ ]:
#XGB: stacked residual correction with rolling-window lambda optimization
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize_scalar
import os

WINDOW = 2000
TUNE_WINDOW = 168
BATCH_SIZE = 24

RESULTS_DIR = "/home/s13q981/HTL/gam_xgb_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Genuine GAM+XGB training-period (2016-2019) predictions, used by MinT/CNN to fit
# their reconciliation weights on held-out history instead of carving a chunk out of
# the test period. Decoupled from the test-period checkpoint below: the initial
# xgb_model fit (gs.fit) is cheap and happens regardless of whether the test
# checkpoint already exists, so already-completed BAs get their training prediction
# filled in without re-running the expensive walk-forward loop. Uses lambda=1.0,
# matching the value the walk-forward loop itself starts from before any tuning.
TRAIN_RESULTS_DIR = "/home/s13q981/HTL/gam_train_predictions"
os.makedirs(TRAIN_RESULTS_DIR, exist_ok=True)

final_results = {}
for ba, data in tqdm(ba_data.items(), desc="Stacking XGB"):
    ckpt_path = f"{RESULTS_DIR}/{ba}.parquet"
    train_ckpt_path = f"{TRAIN_RESULTS_DIR}/{ba}.parquet"
    test_done = os.path.exists(ckpt_path)
    train_done = os.path.exists(train_ckpt_path)
    if test_done and train_done:
        continue  # both already computed, skip entirely -- makes restarts cheap

    df_train_ba, df_test_ba = data["df_train"], data["df_test"]
    X_train_feat = df_train_ba[FEATURES].astype(float)
    X_test_feat  = df_test_ba[FEATURES].astype(float)
    gam_pred_train, gam_pred_test = data["gam_pred_train"], data["gam_pred_test"]
    y_train_actual, y_test_actual = data["y_train"], data["y_test"]
    residuals_train = (y_train_actual - gam_pred_train).astype(float)
    X_train_xgb = X_train_feat.copy()
    X_train_xgb["gam_pred"] = gam_pred_train
    gs = GridSearchCV(
        estimator=xgb.XGBRegressor(objective="reg:squarederror", random_state=42),
        param_grid={"n_estimators": [100, 200], "max_depth": [3, 4]},
        cv=3, scoring="neg_mean_squared_error"
    )
    gs.fit(X_train_xgb, residuals_train)
    xgb_model = gs.best_estimator_

    if not train_done:
        xgb_pred_train = xgb_model.predict(X_train_xgb)
        pred_train = gam_pred_train + 1.0 * xgb_pred_train
        df_train_result_ba = df_train_ba[["datetime_utc", "balancing_authority_code_eia"]].copy()
        df_train_result_ba["y_true"] = y_train_actual
        df_train_result_ba["pred_train"] = pred_train
        df_train_result_ba.to_parquet(train_ckpt_path, index=False)

    if test_done:
        continue  # test checkpoint already exists, skip the expensive walk-forward

    current_X_history = X_train_xgb.copy()
    current_res_history = list(residuals_train)
    X_test_xgb = X_test_feat.copy()
    X_test_xgb["gam_pred"] = gam_pred_test
    updated_preds, lambda_history = [], []
    lambda_val = 1.0
    for i in range(0, len(X_test_xgb), BATCH_SIZE):
        X_batch = X_test_xgb.iloc[i:i + BATCH_SIZE]
        y_batch_actual = y_test_actual[i:i + BATCH_SIZE]
        gam_batch_pred = gam_pred_test[i:i + BATCH_SIZE]

        #predict next 24 hours
        res_pred = xgb_model.predict(X_batch)
        updated_preds.extend(gam_batch_pred + (lambda_val * res_pred))
        lambda_history.extend([lambda_val] * len(X_batch))

        #apply residuals, roll window: drop oldest 24h, add newest 24h
        observed_residuals = y_batch_actual - gam_batch_pred
        current_res_history.extend(observed_residuals.tolist())
        current_X_history = pd.concat([current_X_history, X_batch]).iloc[-WINDOW:]
        current_res_history = current_res_history[-WINDOW:]

        #split window into 2 pieces, 1 refits 1 tunes lambda
        X_train_window = current_X_history.iloc[:-TUNE_WINDOW] #2000-168h
        res_train_window = current_res_history[:-TUNE_WINDOW]
        X_tune_window = current_X_history.iloc[-TUNE_WINDOW:] #most recent 168h
        res_tune_window = np.array(current_res_history[-TUNE_WINDOW:], dtype=float)
        xgb_model.fit(X_train_window, res_train_window) #refit XGB on last 1832h

        #refit XGB on last 168h
        xgb_preds_tune = xgb_model.predict(X_tune_window) #evaluate
        def objective_function(l):
            return np.mean((res_tune_window - (l * xgb_preds_tune)) ** 2)
        opt_result = minimize_scalar(objective_function, bounds=(0.0, 1.0), method="bounded")
        lambda_val = opt_result.x #tune lambda
    y_pred = np.array(updated_preds, dtype=float)
    y_true = y_test_actual[:len(y_pred)]

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))

    df_result_ba = df_test_ba.iloc[:len(y_pred)][
        ["datetime_utc", "balancing_authority_code_eia", "subperiod"]
    ].copy()
    df_result_ba["y_true"] = y_true
    df_result_ba["gam_pred"] = gam_pred_test[:len(y_pred)]
    df_result_ba["y_pred"] = y_pred
    df_result_ba["lambda"] = lambda_history[:len(y_pred)]

    df_result_ba.to_parquet(ckpt_path, index=False)

    final_results[ba] = {
        "rmse": rmse,
        "mae": mae,
        "df_result": df_result_ba,
    }

    print(f"{ba}: RMSE={rmse:.2f}  MAE={mae:.2f}")

print(f"\nFinished stacking XGB for {len(final_results)} / {len(ba_data)} BAs")